In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain.agents import create_agent
from model_config import deepseek,qwen
from tools import get_weather,web_search


In [ ]:
connection = sqlite3.connect("resources/checkpoint.db", check_same_thread=False)
checkpointer = SqliteSaver(connection)
checkpointer.setup()

In [ ]:
config = {"configurable": {"thread_id": "thread_1"}}
agent = create_agent(
    model = qwen,
    tools = [web_search],
    system_prompt = "你是一个智能助手",
    # system_prompt = system_prompt,
    # response_format= AnswerInfo,
    checkpointer=checkpointer
  )

In [ ]:
response = agent.invoke(
    {
    "messages": [
        HumanMessage(content="我是谁 ")
    ]
    },
    config
)
for message in response['messages']:
    print(message.pretty_print())

记忆管理策略--摘要

In [15]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from model_config import deepseek,qwen
from tools import get_weather,web_search

In [16]:
middlewares = SummarizationMiddleware(
    model=qwen,
    trigger=('messages',3),
    # trigger=("fraction", 0.6),
    # trigger=("tokens", 4000),
    keep=('messages',1)
)

In [17]:
connection = sqlite3.connect("resources/checkpoint.db", check_same_thread=False)
checkpointer = SqliteSaver(connection)
checkpointer.setup()

In [19]:
config = {"configurable": {"thread_id": "thread_25"}}
agent = create_agent(
    model = qwen,
    tools = [web_search],
    system_prompt = "你是一个智能助手",
    # system_prompt = system_prompt,
    # response_format= AnswerInfo,
    checkpointer=checkpointer,
    middleware=[middlewares]
    
  )